# Training eines neuronalen Netzes zur Erkennung von handschriftlichen Ziffern

## 0) Ziel
Es soll ein KI-System entwickelt werden, das mithilfe eines neuronalen Netzes handschriftliche Ziffern erkennt. Das System könnte z. B. bei der automatisierten Erkennung und Digitalisierung von Ziffern in folgenden Anwendungsfällen eingesetzt werden:
* Note auf einer Schülerarbeit oder von handschriftlichen Notenbögen
* Postleitzahlen auf Briefen
* handschriftliche Steuerformulare oder Überweisungsträger
* Handschrifterkennung auf einem Tablet

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 1</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Diskutieren Sie (insgesamt maximal 10 Minuten) für zwei Anwendungsfälle die Chancen und Risiken des Einsatzes eines solchen KI-Systems. Erläutern Sie dabei insbesondere die Auswirkungen für den Fall, dass eine bestimmte Ziffer falsch erkannt wird.</li>
    <li style="list-style-type: lower-alpha;">Ein KI-System wird nie zu 100% korrekt klassifizieren können. Geben Sie für alle gegebenen Anwendungsfälle einen intuitiven Wert für die Genauigkeit an, den Sie erwarten würden, um das System einzusetzen. Notieren Sie sich die Werte oben hinter den Anwendungsfällen.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

# Übersicht
Dieses Jupyter Notebook erarbeitet schrittweise die Erstellung eines neuronalen Netzes. Dazu werden folgende Schritte durchlaufen


1.   Importieren der notwendigen Bibliotheken
2.   Laden der Trainings- und Testdaten
3.   Analyse der Daten
4.   Vorbereitung der Daten fürs Training
5.   Neuronales Netz definieren
6.   Neuronales Netz trainieren
7.   Netz mit Testdaten testen
8.   Netz auf neue Daten anwenden
9.   Ausblick auf komplexere Netze




## 1) Importieren der notwendigen Bibliotheken
Das neuronale Netz wird mit scikit-learn erstellt. Scikit-learn ist eine weit verbreitete Bibliothek für maschinelles Lernen in Python. Numpy und matplotlib dienen für mathematische Berechnungen sowie zur Visualisierung. PIL wird ganz am Ende der Einheit verwendet, um ein Bild im jpeg-Format so umzuwandeln, dass es vom neuronalen Netz klassifiziert werden kann.
<br><b>Hinweis:</b> Die erste Ausführung kann etwas dauern, da die Bibliotheken im Browser installiert werden.


In [ ]:
# Bibliotheken installieren (nur nötig in JupyterLite)
import micropip
await micropip.install(['scikit-learn', 'matplotlib', 'Pillow'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, log_loss
from PIL import Image

## 2) Daten laden

Der MNIST-Datensatz wird geladen. Er besteht aus handgeschriebenen Ziffern. Der Datensatz enthält zwei Teile: Trainingsdaten und Testdaten. Jeder Teil enthält zwei Elemente:
* Die eigentlichen Bilddaten: Für jedes Bild werden die Grauwerte der Pixel gespeichert.
* Ein Array, das die Labels zu den Bildern enthält.

**Hinweis:** Das Laden kann im Browser ca. 30 Sekunden dauern.

In [ ]:
# MNIST-Datensatz laden
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')

# Daten und Labels aufteilen
data = mnist.data.astype('float32')
labels = mnist.target.astype('int')

# In Trainings- und Testdaten aufteilen (60000 Training, 10000 Test)
train_data = data[:60000]
test_data = data[60000:]
train_labels = labels[:60000]
test_labels = labels[60000:]

# Für die Visualisierung: Bilder als 28x28 Matrizen
train_images = train_data.reshape(-1, 28, 28)
test_images = test_data.reshape(-1, 28, 28)

print(f'Trainingsdaten: {len(train_data)} Bilder')
print(f'Testdaten: {len(test_data)} Bilder')

## 3) Analyse des Datensatzes
Da das Ergebnis eines neuronalen Netzes von den Trainingsdaten abhängt, ist es wichtig, sich zunächst mit dem Datensatz vertraut zu machen. Dazu wird dieser im folgenden Abschnitt anhand von Beispielen visualisiert und anschließend hinsichtlich Aspekte wie der Länge von Trainings- und Testdaten sowie der Verteilung der unterschiedlichen Ziffern analysiert.

Zunächst werden die Grauwerte einer Ziffer aus dem Trainingsdatensatz angezeigt. Anschließend wird diese Ziffer als Bild dargestellt.

In [ ]:
print(f"Das Label des Beispielbildes ist: {train_labels[0]}")
#Grauwerte als Matrix ausgeben
for row in train_images[0]:
    print(" ".join(f"{pixel:3.0f}" for pixel in row))

In [ ]:
# Bild anzeigen
plt.imshow(train_images[0], cmap='gray')  # Bild im Graustufenmodus anzeigen
plt.title(f"Label: {train_labels[0]}")  # Titel mit Label des Bildes
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 2</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Die Anzahl der Neuronen in der Eingabeschicht des neuronalen Netzes soll der Anzahl der Pixel eines Bildes entsprechen. Ermitteln Sie daher (z. B. anhand der Visualisierung am Anfang des Abschnitts) die Anzahl der Pixel eines Bildes?</li>
    <li style="list-style-type: lower-alpha;">Lassen Sie sich andere Ziffern aus dem Trainingsdatensatz anzeigen. Verändern Sie dazu entweder den obigen Quelltext oder fügen Sie unterhalb dieses Arbeitsauftrags eine neue Zelle hinzu.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Um ein Gefühl für die Darstellung der Ziffer 1 zu bekommen, werden die ersten 21 Vorkommen der Ziffer 1 visualisiert.

In [ ]:
# Indizes der Bilder mit der Ziffer 1 finden
indices_of_ones = np.where(train_labels == 1)[0]

# Visualisiere die ersten 21 Vorkommen der Ziffer 1
plt.figure(figsize=(21, 7))
for i in range(21):
    plt.subplot(3, 7, i + 1)
    plt.imshow(train_images[indices_of_ones[i]], cmap='gray')
    plt.title(f"Label: 1")
    plt.axis('off')
plt.tight_layout()
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 3</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Decken sich die Bilder der Ziffer 1 mit Ihren Erwartungen? Wie zeichnen Sie die 1?</li>
     <li style="list-style-type: lower-alpha;">Erläutern Sie, welche Probleme unterschiedliche Zeichnungen einer Ziffer für das Training sowie den Einsatz des Systems mit sich bringen können.</li>
    <li style="list-style-type: lower-alpha;">Lassen Sie sich eine andere Ziffer anzeigen und überprüfen Sie die Ausgabe mit Ihren Erwartungen. Verändern Sie dazu entweder den Quelltext oben oder fügen Sie unterhalb dieses Arbeitsauftrages eine neue Zelle ein.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Um ein Gefühl für die Darstellung aller Ziffern zu bekommen, wird im Folgenden eine durchschnittliche Darstellung jeder Ziffer angezeigt.

In [ ]:
avg_images = []
for digit in range(10):
    digit_images = train_images[train_labels == digit]
    avg_image = np.mean(digit_images, axis=0)
    avg_images.append(avg_image)

# Visualisierung der Durchschnittsbilder
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(avg_images[i], cmap='gray')
    plt.title(f'durchschnittliche {i}')
    plt.axis('off')
plt.tight_layout()
plt.show()

Durch die Visualisierungen haben Sie einen intuitiven Einblick in die Trainingsdaten erhalten. Dieser soll jetzt durch eine statistische Analyse einfacher Aspekte erweitert werden.

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 4</b></p>

Finden Sie heraus, wie viele Trainingsdaten und wie viele Testdaten im Datensatz enthalten sind und berechnen Sie jeweils den prozentualen Anteil am gesamten Datensatz. Verwenden Sie die Zelle unterhalb des Arbeitsauftrages.
<br>(Hinweis 1: Verwenden Sie die Python-Funktion len(s))<br>
<br>
<details>
<summary>Klicken für Hinweis 2</summary>
<p>Die Trainings- und Testdaten können der Funktion len(s) als Parameter übergeben werden.</p>
</details>
<p style="background-color: yellow; text-align:center">&nbsp</p>

In [ ]:
# Arbeitsauftrag: Länge von Trainingsdaten und Testdaten sowie Verhältnis


Es wird überprüft, ob die Daten tatsächlich 10 unterschiedliche Ziffern enthalten.

In [ ]:
unique_labels = np.unique(train_labels)
print(f"Folgende Label werden in den Trainingsdaten vergeben: {unique_labels}")

Wie ist die Verteilung der Ziffern in den Trainingsdaten?

In [ ]:
digit_counts = np.bincount(train_labels)
print(f"Häufigkeit der Ziffern im Trainingsdatensatz:")
for digit, count in enumerate(digit_counts):
    print(f"Ziffer {digit}: {count} Vorkommen")

Diese Häufigkeit kann man auch anschaulich visualisieren:

In [ ]:
plt.bar(range(10), digit_counts)
plt.xlabel('Ziffer')
plt.ylabel('Anzahl der Vorkommen')
plt.title('Verteilung der Ziffern im Trainingsdatensatz')
plt.xticks(range(10))
plt.show()

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 5</b>
</p>

Überprüfen Sie, ob die einzelnen Ziffern in den Testdaten genauso häufig vorkommen wie in den Trainingsdaten. Ergänzen Sie dazu entweder den Quelltext oben oder fügen Sie unterhalb dieses Arbeitsauftrages eine neue Zelle hinzu.
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 4) Daten vorbereiten
Damit die Daten zum Training geeignet sind, müssen sie vorbereitet werden:
<ol>
<li>Sie müssen normalisiert werden.
<li>Die Bilddaten müssen in eindimensionale Vektoren umgewandelt werden ("Flatten").
</ol>

1) Für das Training müssen die Daten normalisiert werden. Überlegen Sie, wie Sie Grauwerte zwischen 0 und 255 so normalisieren können, dass sie aus Zahlen zwischen 0 und 1 bestehen.
Ergänzen Sie anschließend den folgenden Quelltext:

In [ ]:
train_data_norm = train_data.astype("float32") / #Ergänzung
test_data_norm = test_data.astype("float32") / #Ergänzung

<details>
<summary>Lösung</summary>
<p>Die Daten müssen durch 255 geteilt werden.</p>
</details>

Zur Überprüfung wird ein normalisiertes Bild ausgegeben.

In [ ]:
print(f"Das Label des Beispielbildes ist: {train_labels[0]}")
#Grauwerte als Matrix ausgeben
for row in train_data_norm[0].reshape(28, 28):
    print(" ".join(f"{pixel:3.2f}" for pixel in row))

2) Die Bilddaten liegen bereits als eindimensionale Vektoren mit 784 Werten vor (28 × 28 Pixel). Das ist genau das Format, das das neuronale Netz als Eingabe erwartet. Jeder Pixel entspricht einem Neuron in der Eingabeschicht.

Überprüfen wir die Form der Daten:

In [ ]:
print(f"Form der Trainingsdaten: {train_data_norm.shape}")
print(f"Jedes Bild ist ein Vektor mit {train_data_norm.shape[1]} Werten (= 28 × 28 Pixel)")

## 5) Neuronales Netz (Modell) definieren
Jetzt wird das eigentliche Modell definiert. Wir verwenden einen MLPClassifier (Multi-Layer Perceptron) aus scikit-learn. Das ist ein neuronales Netz mit vollständig verbundenen Schichten.

Das Netz hat folgende Architektur:
- **Eingabeschicht:** 784 Neuronen (ein Neuron pro Pixel)
- **Verborgene Schicht:** 100 Neuronen mit ReLU-Aktivierungsfunktion
- **Ausgabeschicht:** 10 Neuronen mit Softmax-Aktivierungsfunktion (eine pro Ziffer 0–9)

In [ ]:
# Neuronales Netz definieren
model = MLPClassifier(
    hidden_layer_sizes=(100,),  # Eine verborgene Schicht mit 100 Neuronen
    activation='relu',          # Aktivierungsfunktion: ReLU
    solver='adam',              # Optimizer: Adam
    batch_size=100,             # Batch-Größe
    max_iter=1,                 # Wird manuell pro Epoche trainiert
    warm_start=True,            # Erlaubt weiteres Training
    random_state=42
)

print('Netz definiert mit:')
print(f'  Eingabeschicht:   784 Neuronen (28×28 Pixel)')
print(f'  Verborgene Schicht: 100 Neuronen (Aktivierung: ReLU)')
print(f'  Ausgabeschicht:   10 Neuronen (Aktivierung: Softmax)')

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 6</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Erklären Sie, warum die Eingabeschicht 784 Neuronen hat. Überlegen Sie sich dazu, welches Format die Trainingsdaten haben.</li>
    <li style="list-style-type: lower-alpha;">Geben Sie die Anzahl der Schichten des neuronalen Netzes an.</li>
    <li style="list-style-type: lower-alpha;">Geben Sie die Aktivierungsfunktionen für die verborgene und die Ausgabeschicht an. Geben Sie zudem den Wertebereich der Funktionen an (recherchieren Sie dazu ggf.).
    <li style="list-style-type: lower-alpha;">Erklären Sie, warum in der Ausgabeschicht die softmax-Funktion und nicht ReLU verwendet wird. Beschreiben Sie dazu, wie die Ausgabe der 10 Neuronen interpretiert wird.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

Man kann sich die Anzahl der Parameter des Netzes berechnen:
- Verborgene Schicht: 784 Eingänge × 100 Neuronen + 100 Schwellenwerte (Bias) = **78.500 Parameter**
- Ausgabeschicht: 100 Eingänge × 10 Neuronen + 10 Schwellenwerte (Bias) = **1.010 Parameter**
- **Gesamt: 79.510 Parameter**

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 7</b></p>

Vollziehen Sie die Anzahl der Parameter jeder Schicht nach. Überlegen Sie sich dazu jeweils die Eingänge jeder Schicht, die Anzahl der Neuronen sowie die Schwellenwerte. (Hinweis: Jede Schicht ist vollständig verbunden, d.h. jedes Neuron ist mit allen Neuronen der vorherigen Schicht verbunden.)
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 6) Modell trainieren
Jetzt kann das Modell trainiert werden. Wir trainieren es über 10 Epochen und beobachten dabei, wie sich Genauigkeit (Accuracy) und Verlust (Loss) entwickeln.

In [ ]:
# Training über mehrere Epochen
epochen = 10
history_loss = []
history_accuracy = []

for epoche in range(1, epochen + 1):
    model.fit(train_data_norm, train_labels)
    
    # Vorhersagen auf Trainingsdaten
    train_pred = model.predict(train_data_norm)
    train_proba = model.predict_proba(train_data_norm)
    
    # Genauigkeit und Verlust berechnen
    acc = accuracy_score(train_labels, train_pred)
    loss = log_loss(train_labels, train_proba)
    
    history_accuracy.append(acc)
    history_loss.append(loss)
    
    # Anzahl der Batch-Updates pro Epoche
    n_batches = len(train_data_norm) // model.batch_size
    print(f'Epoche {epoche:2d}/{epochen} - {n_batches} Batches - loss: {loss:.4f} - accuracy: {acc:.4f}')

print('\nTraining abgeschlossen!')

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 8</b></p>
<ol>
   <li style="list-style-type: lower-alpha;">Erklären Sie, warum in einer Epoche 600-mal Anpassungen der Parameter stattfinden. Überlegen Sie sich dazu einen Zusammenhang zwischen der Anzahl der Trainingsdaten und der batch_size.</li>
    <li style="list-style-type: lower-alpha;">Beschreiben Sie die Bedeutung der Werte accuracy sowie loss und analysieren Sie die Werte im Verlauf des Trainings.</li>
    <li style="list-style-type: lower-alpha;">Experimentieren Sie mit verschiedenen Werten für batch_size sowie epochs und vergleichen Sie die Ergebnisse. <strong>Führen Sie bei Veränderungen immer erst erneut den Quelltext zur Definition (!) des Netzes aus, bevor Sie trainieren!</strong></li>
    <li style="list-style-type: lower-alpha;">Verändern Sie auch das Netz (z. B. mehr Neuronen in der verborgenen Schicht oder mehr verborgene Schichten, z.B. hidden_layer_sizes=(100, 50)) und vergleichen Sie das Ergebnis.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 7) Modell mit Testdaten testen
Zum Abschluss wird das Modell mit den Testdaten getestet.

In [ ]:
# Modell auf Testdaten evaluieren
test_pred = model.predict(test_data_norm)
test_proba = model.predict_proba(test_data_norm)

test_acc = accuracy_score(test_labels, test_pred)
test_loss = log_loss(test_labels, test_proba)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

<p style= "background-color: yellow;text-align:center"><b>ARBEITSAUFTRAG 9</b></p>

Ganz am Anfang der Einheit haben Sie einen Wunsch für die Genauigkeit des Systems in unterschiedlichen Kontexten angegeben. Vergleichen Sie Ihre Einschätzung mit der aktuellen Genauigkeit des vorliegenden Systems. Recherchieren Sie ggf., welche Genauigkeit aktuelle KI-Systeme bei der Handschriftenerkennung erreichen.
<br><br>
<p style="background-color: yellow; text-align:center">&nbsp</p>

## 8) Modell verwenden
Mit dem folgenden Quelltext werden Bilder aus den Testdaten mit dem Modell klassifiziert und die Vorhersagen visualisiert.

In [ ]:
# Einige zufällige Testbilder klassifizieren
n_examples = 5
indices = np.random.choice(len(test_data_norm), n_examples, replace=False)

sample_data = test_data_norm[indices]
sample_labels = test_labels[indices]
predictions = model.predict_proba(sample_data)

# Visualisierung der Bilder mit Vorhersagen
fig, axes = plt.subplots(1, n_examples, figsize=(12, 3))

for i, ax in enumerate(axes):
    ax.imshow(sample_data[i].reshape(28, 28), cmap='gray')
    predicted_class = np.argmax(predictions[i])
    confidence = predictions[i][predicted_class] * 100
    ax.set_title(f'Vorhergesagt: {predicted_class}\n({confidence:.1f}%)\nWahr: {sample_labels[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

<p style="background-color: yellow; text-align:center"><b>ARBEITSAUFTRAG 10</b></p>
<ol>
    <li style="list-style-type: lower-alpha;">Führen Sie die obige Zelle mehrmals aus, um verschiedene Testbilder zu sehen. Werden alle Ziffern korrekt erkannt?</li>
    <li style="list-style-type: lower-alpha;">Untersuchen Sie, bei welchen Ziffern das Netz am häufigsten Fehler macht. Stellen Sie eine Vermutung darüber auf, warum bestimmte Ziffern häufiger verwechselt werden.</li>
</ol>
<p style="background-color: yellow; text-align:center">&nbsp</p>

### Fehleranalyse
Schauen wir uns an, welche Ziffern am häufigsten falsch klassifiziert werden:

In [ ]:
# Confusion Matrix visualisieren
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, test_pred)

plt.figure(figsize=(10, 8))
plt.imshow(cm, interpolation='nearest', cmap='Blues')
plt.title('Confusion Matrix')
plt.colorbar()
plt.xlabel('Vorhergesagte Ziffer')
plt.ylabel('Tatsächliche Ziffer')
plt.xticks(range(10))
plt.yticks(range(10))

# Zahlen in die Zellen schreiben
for i in range(10):
    for j in range(10):
        plt.text(j, i, str(cm[i, j]), ha='center', va='center',
                 color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

## 9) Ausblick
In der Praxis würde ein neuronales Netz zur Erkennung von handschriftlichen Ziffern deutlich komplexer aussehen. Hier testen wir ein tieferes Netz mit **zwei verborgenen Schichten** (200 und 100 Neuronen). Mehr Schichten und Neuronen erlauben es dem Netz, komplexere Muster zu lernen.

Bei der Bilderkennung werden üblicherweise sogenannte Convolutional-Netzwerke (CNNs) eingesetzt, die speziell darauf ausgelegt sind, räumliche Muster in Bildern zu erkennen. Diese können jedoch nicht direkt im Browser ausgeführt werden und erfordern spezialisierte Bibliotheken wie TensorFlow oder PyTorch.

Trainieren Sie das tiefere Netz und vergleichen Sie die Genauigkeit mit der des einfachen Netzes.

In [ ]:
# Tieferes Netz mit zwei verborgenen Schichten
model_deep = MLPClassifier(
    hidden_layer_sizes=(200, 100),  # Zwei verborgene Schichten: 200 und 100 Neuronen
    activation='relu',
    solver='adam',
    batch_size=100,
    max_iter=1,
    warm_start=True,
    random_state=42
)

# Training
epochen = 10
for epoche in range(1, epochen + 1):
    model_deep.fit(train_data_norm, train_labels)
    train_pred = model_deep.predict(train_data_norm)
    acc = accuracy_score(train_labels, train_pred)
    print(f'Epoche {epoche:2d}/{epochen} - accuracy: {acc:.4f}')

print('\nTraining abgeschlossen!')

In [ ]:
# Tieferes Modell evaluieren
test_pred_deep = model_deep.predict(test_data_norm)
test_acc_deep = accuracy_score(test_labels, test_pred_deep)

print(f"Test accuracy (einfaches Netz):  {test_acc:.4f}")
print(f"Test accuracy (tieferes Netz):   {test_acc_deep:.4f}")